In [1]:
# Importing the required libraries
import keras
import numpy as np
from keras.models import Sequential,Model
from keras.layers import Dense,Bidirectional
from nltk.tokenize import word_tokenize,sent_tokenize
from keras.layers import *
from sklearn.model_selection import cross_val_score
import nltk
import pandas as pd
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df=pd.read_csv('/content/drive/MyDrive/Emotion_Prediction_GeeksForGeeks/isear.csv',header=None)
df.drop(df[df[1] == '[ No response.]'].index, inplace = True)

In [4]:
df.head()

,0,1
0,joy,[ On days when I feel close to my partner and ...
1,fear,Every time I imagine that someone I love or I ...
2,anger,When I had been obviously unjustly treated and...
3,sadness,When I think about the short time that we live...
4,disgust,At a gathering I found myself involuntarily si...


In [5]:
nltk.download('punkt')
nltk.download('punkt_tab')

# 1. Extract sentences and tokenize them
feel_arr = df[1].apply(word_tokenize).tolist()

print(f"First tokenized sentence before padding: {feel_arr[0]}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


First tokenized sentence before padding: ['[', 'On', 'days', 'when', 'I', 'feel', 'close', 'to', 'my', 'partner', 'and', 'other', 'friends', '.', 'When', 'I', 'feel', 'at', 'peace', 'with', 'myself', 'and', 'also', 'experience', 'a', 'close', 'contact', 'with', 'people', 'whom', 'I', 'regard', 'greatly', '.', ']']


In [6]:
# 2. Define and apply padding function to ensure uniform sentence length
max_sentence_length = 100

def pad_sentence(arr, max_len):
    if len(arr) < max_len:
        return arr + ['<pad>'] * (max_len - len(arr))
    return arr[:max_len]

feel_arr = [pad_sentence(sent, max_sentence_length) for sent in feel_arr]

print(f"First tokenized sentence after padding: {feel_arr[0]}")
print(f"Length of first padded sentence: {len(feel_arr[0])}")

First tokenized sentence after padding: ['[', 'On', 'days', 'when', 'I', 'feel', 'close', 'to', 'my', 'partner', 'and', 'other', 'friends', '.', 'When', 'I', 'feel', 'at', 'peace', 'with', 'myself', 'and', 'also', 'experience', 'a', 'close', 'contact', 'with', 'people', 'whom', 'I', 'regard', 'greatly', '.', ']', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Length of first padded sentence: 100


In [7]:
# 3. Load GloVe Embeddings
vocab_f = '/content/drive/MyDrive/glove.6B.50d.txt'
embeddings_index = {}
with open(vocab_f, encoding='utf8') as f:
    for line in f:
        values = line.rstrip().rsplit(' ')
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Loaded {len(embeddings_index)} word vectors.")
print(f"Embedding for 'happy': {embeddings_index.get('happy')[:5]}...")

Loaded 400000 word vectors.
Embedding for 'happy': [ 0.092086  0.2571   -0.58693  -0.37029   1.0828  ]...


In [8]:
# 4. Create embedded_feel_arr (numerical representation of sentences)
embedded_feel_arr = []
embedding_dim = 50

for each_sentence in feel_arr:
    sentence_embeddings = []
    for word in each_sentence:
        if word.lower() in embeddings_index:
            sentence_embeddings.append(embeddings_index[word.lower()])
        else:
            sentence_embeddings.append(np.zeros(embedding_dim, dtype=np.float32))
    embedded_feel_arr.append(sentence_embeddings)

print(f"Shape of embedded_feel_arr (first sentence, first word): {embedded_feel_arr[0][0].shape}")
print(f"Example embedding (first word of first sentence): {embedded_feel_arr[0][0][:5]}...")

Shape of embedded_feel_arr (first sentence, first word): (50,)
Example embedding (first word of first sentence): [-0.61201   0.98226   0.11539   0.014623  0.23873 ]...


In [9]:
# 5. Convert embedded_feel_arr to a 3D NumPy array (X)
X = np.array(embedded_feel_arr, dtype=np.float32)

print(f"Shape of X: {X.shape}")
print(f"X dtype: {X.dtype}")

Shape of X: (7575, 100, 50)
X dtype: float32


In [10]:
from sklearn.preprocessing import OneHotEncoder

# 6. One-hot encode the emotions (Y)
enc = OneHotEncoder(handle_unknown='ignore')
Y = enc.fit_transform(np.array(df[0]).reshape(-1, 1)).toarray()

print(f"Shape of Y: {Y.shape}")
print(f"Y dtype: {Y.dtype}")
print(f"First 5 one-hot encoded labels: {Y[:5]}")

Shape of Y: (7575, 7)
Y dtype: float64
First 5 one-hot encoded labels: [[0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0.]]


In [11]:
from sklearn.model_selection import train_test_split

# 7. Split data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of Y_train: {Y_train.shape}")
print(f"X_train dtype: {X_train.dtype}")
print(f"Y_train dtype: {Y_train.dtype}")

Shape of X_train: (6060, 100, 50)
Shape of Y_train: (6060, 7)
X_train dtype: float32
Y_train dtype: float64


In [12]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Input, Bidirectional, LSTM, Dense, Dropout

def build_bilstm_model(input_shape, output_size):
  m = Sequential()
  m.add(Input(shape=input_shape, dtype=tf.float32))
  m.add(Bidirectional(LSTM(100)))
  m.add(Dropout(0.5))
  m.add(Dense(output_size, activation='softmax'))
  m.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
  return m

print("Model definition function 'build_bilstm_model' is ready.")

Model definition function 'build_bilstm_model' is ready.


In [13]:

X_train_tf = tf.convert_to_tensor(X_train, dtype=tf.float32)
Y_train_tf = tf.convert_to_tensor(Y_train, dtype=tf.float32)

input_shape_for_model = (X_train_tf.shape[1], X_train_tf.shape[2])
output_size_for_model = Y_train_tf.shape[1]

bilstmModel = build_bilstm_model(input_shape_for_model, output_size_for_model)

print("Starting model training...")
bilstmModel.fit(X_train_tf, Y_train_tf, epochs=32, batch_size=128)

Starting model training...
Epoch 1/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.2228 - loss: 1.9054
Epoch 2/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3335 - loss: 1.7309
Epoch 3/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3805 - loss: 1.6393
Epoch 4/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4054 - loss: 1.5663
Epoch 5/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4266 - loss: 1.5377
Epoch 6/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4408 - loss: 1.5084
Epoch 7/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4568 - loss: 1.4677
Epoch 8/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4601 - loss: 1.4558
Epoch 9/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.4710 - loss: 1.4210
Epoch 10/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4856 - loss: 1.3962
Epoch 11/32
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4988 - loss: 1.3544
Epoch 12/32
48/48 ━━━━━━━━━━━━━━━━━━

In [14]:

X_test_tf = tf.convert_to_tensor(X_test, dtype=tf.float32)
Y_test_tf = tf.convert_to_tensor(Y_test, dtype=tf.float32)

print("Evaluating model on test data...")
loss, accuracy = bilstmModel.evaluate(X_test_tf, Y_test_tf)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Evaluating model on test data...
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5017 - loss: 1.4897
Test Loss: 1.4897
Test Accuracy: 0.5017


In [15]:
keras_save_path = '/content/drive/MyDrive/bilstm_emotion_model.keras'
bilstmModel.save(keras_save_path)
print(f"Model saved to {keras_save_path}")

h5_save_path = '/content/drive/MyDrive/bilstm_emotion_model.h5'
bilstmModel.save(h5_save_path)
print(f"Model saved to {h5_save_path}")

Model saved to /content/drive/MyDrive/bilstm_emotion_model.keras
Model saved to /content/drive/MyDrive/bilstm_emotion_model.h5
